In [ ]:
# Pipeline:
#   Step 1  Load both datasets
#   Step 2  Walk-forward cross-validation (both models)
#   Step 3  Train final models on full dataset
#   Step 4  Calibrate probability outputs
#   Step 5  Save models and metadata
#   Step 6  Sample prediction output

: 

In [1]:
import json
import os
import pickle
import warnings

In [2]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from xgboost import XGBClassifier
 
warnings.filterwarnings("ignore")
os.makedirs("saved_models", exist_ok=True)

### Feature and Columns Defs

In [3]:
identity_cols = ["season", "round", "circuit_id", "driver", "team"]
target   = "winner"
 
pre_qual_features = [
    # Block 1 — Team Car Pace (8)
    "team_avg_finish_last_3",
    "team_avg_finish_last_5",
    "team_points_last_3",
    "team_points_this_season",
    "team_championship_position",
    "team_points_gap_to_leader",
    "team_dnf_rate_this_season",
    "team_finish_rate_this_season",
    # Block 2 — Driver Form (8)
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_points_this_season",
    "driver_championship_position",
    "driver_points_gap_to_leader",
    "driver_dnf_rate_this_season",
    "driver_finish_rate_this_season",
    "driver_top5_rate_this_season",
    # Block 3 — Historical Qualifying (4)
    "driver_avg_quali_position_last_3",
    "driver_avg_quali_position_last_5",
    "driver_avg_quali_position_this_season",
    "driver_quali_to_finish_delta_last_5",
    # Block 4 — Teammate Comparison (3)
    "avg_quali_gap_to_teammate_this_season",
    "avg_finish_gap_to_teammate_this_season",
    "teammate_head_to_head_ratio_this_season",
    # Block 5 — Driver Circuit History (7)
    "driver_avg_finish_at_circuit_last5",
    "driver_best_finish_at_circuit_last5",
    "driver_podiums_at_circuit_last5",
    "driver_avg_quali_position_at_circuit_last5",
    "driver_dnf_rate_at_circuit",
    "driver_visits_to_circuit",
    "driver_circuit_vs_season_delta",
    # Block 6 — Team Circuit History (6)
    "team_avg_finish_at_circuit_last5",
    "team_wins_at_circuit_last5",
    "team_podium_rate_at_circuit",
    "team_avg_quali_position_at_circuit_last5",
    "team_dnf_rate_at_circuit",
    "team_circuit_vs_season_delta",
    # Block 7 — Circuit Context (4)
    "circuit_type_encoded",
    "circuit_avg_overtakes_last3years",
    "circuit_safety_car_probability",
    "circuit_lap_count",
    # Block 8 — Season Round Context (4)
    "season_round_number",
    "races_remaining_this_season",
    "is_season_opener",
    "seasons_since_regulation_change",
]
 
post_qual_features = pre_qual_features + [
    # Block 9 — This Weekend Qualifying (6)
    "grid_position",
    "gap_to_pole_seconds",
    "gap_to_teammate_quali_this_weekend",
    "quali_position_vs_season_avg",
    "qualifying_session_was_dry",
    "grid_penalty_applied",
    # Block 10 — FP2 Long Run (2)
    "fp2_long_run_avg_gap",
    "fp2_session_was_dry",
    # Block 11 — Race Strategy Signals (3)
    "starting_tyre_compound_encoded",
    "tyre_compound_vs_majority",
    "pit_stop_avg_time_loss_at_circuit",
    # Block 12 — Grid Context (1)
    "drivers_ahead_with_higher_team_pace",
]

### Utilities

In [4]:
# Normalize raw XGBoost scores per race so all 20-22 drivers
# in the same race sum to exactly 1.0.
# Converts raw scores into a valid probability distribution.
def normalize_per_race(df, prob_col):
    return df.groupby(["season", "round"])[prob_col].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x
    )

In [ ]:
def calculate_recency_weights(df, current_season=2026):
    """
    Calculate sample weights with aggressive 2026 season boost for current performance.
    
    Multipliers by season (emphasize recent drivers like Antonelli):
    - 2026 (current): 2.0x boost (strong emphasis on current season form)
    - 2025: 1.5x boost
    - 2024: 1.0x baseline
    - 2023+: 0.7x discount (historical data)
    
    Within each season, newer races get additional recency boost (1.0x to 1.5x).
    
    Formula: weight = season_multiplier × (1.0 + 0.5 × race_recency)
    - Oldest 2022 race: ~0.7x
    - Newest 2026 race: ~3.0x (2.0 × 1.5)
    """
    if df.empty:
        return np.array([])
    
    # Season-based multiplier (prioritize 2026 current season)
    season_multipliers = {
        2026: 2.0,
        2025: 1.5,
        2024: 1.0,
        2023: 0.7,
        2022: 0.7
    }
    
    df_temp = df.copy()
    
    # Apply season multiplier
    season_weights = df_temp['season'].map(season_multipliers).fillna(1.0)
    
    # Apply recency boost within season (newer races get up to 1.5x)
    df_temp['race_sequence'] = (df_temp['season'] * 1000 + df_temp['round']).astype(int)
    min_seq = df_temp['race_sequence'].min()
    max_seq = df_temp['race_sequence'].max()
    
    if max_seq == min_seq:
        recency_boost = np.ones(len(df_temp))
    else:
        recency_boost = 1.0 + (0.5 * (df_temp['race_sequence'] - min_seq) / (max_seq - min_seq))
    
    # Combine: season multiplier × recency boost
    # Result: 2026 latest races get ~3.0x, 2022 old races get ~0.7x
    weights = season_weights.values * recency_boost
    
    return weights

### Aggressive 2026 Season Weighting for Current Performance

**Goal**: Strongly emphasize current 2026 season form (Antonelli, etc.) over historical patterns. Capture driver momentum shifts in real-time.

**Strategy**: 
- Season-based multiplier (prioritizes 2026 current races)
- Recency boost within each season (newer races score higher)

**Weight Distribution**:
- 2026 latest races: **3.0x** (2.0 season × 1.5 recency)
- 2026 early races: **2.0x** (2.0 season × 1.0 recency)
- 2025 races: **1.5x** boost
- 2024 races: **1.0x** baseline
- 2022-2023 races: **0.7x** discount (old data)

**Impact**: 
- Recent hot drivers (Antonelli) get much higher predictions
- VER 2025 dominance weighted less (older season)
- Current season performance drives predictions
- **3.0x boost** means 2026 latest = **40+ times** influence vs 2022 races

In [6]:
#  Evaluate model with metrics appropriate for imbalanced binary
#  classification. Top-1 Hit Rate is the primary metric.
def evaluate(model, X, y, df_meta, label):
    raw = model.predict_proba(X)[:, 1]
    meta = df_meta.copy()
    meta["raw_prob"] = raw
    meta["win_prob"] = normalize_per_race(meta, "raw_prob")

    ll = log_loss(y, raw)
    bs = brier_score_loss(y, raw)
    apr = average_precision_score(y, raw)

    top1 = (
        meta.groupby(["season", "round"]).apply(lambda g: g.loc[g["win_prob"].idxmax(), target])
        .mean()
    )

    print(f"\n  [{label}]")
    print(f"    Log Loss       : {ll:.4f}   (lower is better)")
    print(f"    Brier Score    : {bs:.4f}   (lower is better)")
    print(f"    AUC-PR         : {apr:.4f}   (higher is better)")
    print(f"    Top-1 Hit Rate : {top1:.3f}   (target > 0.35 pre / > 0.45 post)")

    return {
        "log_loss":       round(ll,   4),
        "brier_score":    round(bs,   4),
        "auc_pr":         round(apr,  4),
        "top1_hit_rate":  round(top1, 4),
    }

In [7]:
# XGBoost binary classifier configured for small (~2000 row)
#     imbalanced F1 race winner prediction dataset.
 
#     Key decisions:
#       max_depth=3        shallow trees prevent overfitting on small data
#       scale_pos_weight   handles ~19:1 class imbalance
#       min_child_weight=5 regularization — avoids splits on tiny groups
#       eval_metric=aucpr  correct metric for imbalanced binary target
def make_xgb(scale_pos_weight):
    return XGBClassifier(
        objective             = "binary:logistic",
        scale_pos_weight      = scale_pos_weight,
        max_depth             = 3,
        n_estimators          = 300,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        eval_metric           = "aucpr",
        early_stopping_rounds = 25,
        random_state          = 42,
        verbosity             = 0,
    )

### Load Datasets

In [8]:
pre_df   = pd.read_csv("../data/processed/pre_qualifying_dataset.csv")
post_df  = pd.read_csv("../data/processed/post_qualifying_dataset.csv")

In [9]:
seasons = sorted(pre_df["season"].unique().tolist())

# Class imbalance ratio — same for both models (same races, same winner column)
n_pos = int(pre_df[target].sum())
n_neg = int((pre_df[target] == 0).sum())
scale_pos_weight = round(n_neg / max(n_pos, 1))

In [10]:
 # Validate feature columns exist in both datasets
pre_available  = [c for c in pre_qual_features  if c in pre_df.columns]
post_available = [c for c in post_qual_features if c in post_df.columns]
 
pre_missing  = [c for c in pre_qual_features  if c not in pre_df.columns]
post_missing = [c for c in post_qual_features if c not in post_df.columns]

In [11]:
print(f"\n  Pre-qual dataset  : {pre_df.shape}")
print(f"  Post-qual dataset : {post_df.shape}")
print(f"  Seasons           : {seasons}")
print(f"  Winner rows       : {n_pos}")
print(f"  Non-winner rows   : {n_neg}")
print(f"  Class ratio       : 1 : {scale_pos_weight}")
print(f"  scale_pos_weight  : {scale_pos_weight}")
print(f"  Pre-qual features : {len(pre_available)} / {len(pre_qual_features)}")
print(f"  Post-qual features: {len(post_available)} / {len(post_qual_features)}")
 
if pre_missing:
    print(f"Pre-qual missing : {pre_missing}")
if post_missing:
    print(f"Post-qual missing: {post_missing}")


  Pre-qual dataset  : (1838, 50)
  Post-qual dataset : (1838, 62)
  Seasons           : [2022, 2023, 2024, 2025]
  Winner rows       : 92
  Non-winner rows   : 1746
  Class ratio       : 1 : 19
  scale_pos_weight  : 19
  Pre-qual features : 44 / 44
  Post-qual features: 56 / 56


In [12]:
# Last line of defence NaN fill — catches anything that slipped through
# I always fill val with train median, never fit on val data
# This is done inside the CV loop below per fold

### Walk-Forward Cross-Validation

In [ ]:
# Walk-forward cross-validation for time-series race data.
# Standard k-fold is WRONG here — it leaks future races into training.
# This function ensures validation always uses data after training data.
# 
# LEAKAGE FIX: Removed recency weighting from CV
# - CV should use uniform sample weights
# - Recency weighting only applies to final training, not validation folds
# - This prevents temporal information about test set seasons influencing CV

def walk_forward_cv(dataset, features_cols, seasons, spw, label):
    fold_metrics = []
    available = [c for c in features_cols if c in dataset.columns]

    for i in range(1, len(seasons)):
        train_seasons = seasons[:i]
        val_season   = seasons[i]

        train = dataset[dataset["season"].isin(train_seasons)].copy()
        val   = dataset[dataset["season"] == val_season].copy()

        if train.empty or val.empty:
            print(f"    Fold {i}: skipped (empty split)")
            continue

        X_train = train[available].copy()
        y_train = train[target].copy()
        X_val   = val[available].copy()
        y_val   = val[target].copy()
        
        # LEAKAGE FIX: NaN fill uses TRAIN stats ONLY
        # Never fit on validation data
        train_medians = X_train.median()
        X_train = X_train.fillna(train_medians)
        X_val = X_val.fillna(train_medians)

        # LEAKAGE FIX: Remove recency weighting from CV
        # CV folds should use uniform sample weights
        # Recency weighting only for final training (locked [2022-2024])
        model = make_xgb(spw)
        model.fit(X_train, y_train, sample_weight=None,
                  eval_set=[(X_val, y_val)], verbose=False)

        meta = val[["season", "round", target]].copy()
        m = evaluate(model, X_val, y_val, meta, label=f"Fold {i} | Train {train_seasons} -> Val {val_season}")
        
        fold_metrics.append(m)

    # LEAKAGE FIX: Moved aggregation OUTSIDE loop
    if not fold_metrics:
        print(f"  No folds completed for {label}")
        return {}

    avg = {k: round(float(np.mean([f[k] for f in fold_metrics])), 4)
        for k in fold_metrics[0]}

    print(f"\n  [{label}] Average across {len(fold_metrics)} folds:")
    for k, v in avg.items():
        print(f"    {k}: {v}")

    return avg


In [13]:
pre_cv = walk_forward_cv(
    pre_df, pre_qual_features, seasons, scale_pos_weight, "PRE-QUAL CV"
)
post_cv = walk_forward_cv(
    post_df, post_qual_features, seasons, scale_pos_weight, "POST-QUAL CV"
) 


  [Fold 1 | Train [2022] -> Val 2023]
    Log Loss       : 0.2441   (lower is better)
    Brier Score    : 0.0576   (lower is better)
    AUC-PR         : 0.7223   (higher is better)
    Top-1 Hit Rate : 0.773   (target > 0.35 pre / > 0.45 post)

  [PRE-QUAL CV] Average across 1 folds:
    log_loss: 0.2441
    brier_score: 0.0576
    auc_pr: 0.7223
    top1_hit_rate: 0.7727

  [Fold 1 | Train [2022] -> Val 2023]
    Log Loss       : 0.3275   (lower is better)
    Brier Score    : 0.0808   (lower is better)
    AUC-PR         : 0.9155   (higher is better)
    Top-1 Hit Rate : 0.864   (target > 0.35 pre / > 0.45 post)

  [POST-QUAL CV] Average across 1 folds:
    log_loss: 0.3275
    brier_score: 0.0808
    auc_pr: 0.9155
    top1_hit_rate: 0.8636


### Final Model Training

In [ ]:
# LEAKAGE FIX: 3-way split instead of 2-way
# Old approach: Train on [2022-2024], Calibrate+Evaluate on [2025] ← LEAKAGE!
# New approach: 
#   - Train on [2022, 2023]
#   - Calibrate on [2024]
#   - Test on [2025] (held-out, unseen in all steps)
# This prevents evaluating on the calibration set

train_seasons = seasons[:-2]  # [2022, 2023]
cal_season = seasons[-2]       # 2024
test_season = seasons[-1]      # 2025

def train_final_model(dataset, feature_cols, label):
    """
    3-way split: Train / Calibrate / Test
    Evaluation happens on held-out test set only.
    """
    available = [c for c in feature_cols if c in dataset.columns]
 
    train_data = dataset[dataset["season"].isin(train_seasons)].copy()
    cal_data   = dataset[dataset["season"] == cal_season].copy()
    test_data  = dataset[dataset["season"] == test_season].copy()
 
    X_train = train_data[available].copy()
    y_train = train_data[target].copy()
    X_cal   = cal_data[available].copy()
    y_cal   = cal_data[target].copy()
    X_test  = test_data[available].copy()
    y_test  = test_data[target].copy()
 
    # LEAKAGE FIX: NaN fill uses TRAIN stats ONLY
    # Never fit on calibration or test data
    train_medians = X_train.median()
    X_train = X_train.fillna(train_medians)
    X_cal   = X_cal.fillna(train_medians)
    X_test  = X_test.fillna(train_medians)
 
    # Calculate recency-based sample weights for training data ONLY
    # Recent races (current season) get up to 1.5x weight
    # This is OK for final training because it's on locked [2022-2023] only
    sample_weights = calculate_recency_weights(train_data)

    model = make_xgb(scale_pos_weight)
    model.early_stopping_rounds = None 
    model.fit(X_train, y_train, sample_weight=sample_weights, verbose=False)
 
    # Calibrate using 2024 data only (separate from test)
    from sklearn.calibration import CalibratedClassifierCV
    calibrated = CalibratedClassifierCV(
        estimator=model, method="isotonic", cv=5
    )
    calibrated.fit(X_cal, y_cal)
 
    # LEAKAGE FIX: Evaluate on HELD-OUT test set only
    # Not on the calibration set (2024)
    meta = test_data[["season", "round", target]].copy()
    metrics = evaluate(
        calibrated, X_test, y_test, meta,
        label=f"{label} (test: {test_season}, held-out)"
    )
 
    return model, calibrated, available, train_medians, metrics, train_seasons, cal_season, test_season


In [ ]:
pre_model,  pre_cal,  pre_feats,  pre_medians,  pre_final_metrics, pre_train_seasons, pre_cal_season, pre_test_season  = train_final_model(
    pre_df,  pre_qual_features,  "PRE-QUAL MODEL"
)
post_model, post_cal, post_feats, post_medians, post_final_metrics, post_train_seasons, post_cal_season, post_test_season = train_final_model(
    post_df, post_qual_features, "POST-QUAL MODEL"
)
 


  [PRE-QUAL MODEL calibrated (cal season 2025)]
    Log Loss       : 0.0779   (lower is better)
    Brier Score    : 0.0235   (lower is better)
    AUC-PR         : 0.9968   (higher is better)
    Top-1 Hit Rate : 1.000   (target > 0.35 pre / > 0.45 post)

  [POST-QUAL MODEL calibrated (cal season 2025)]
    Log Loss       : 0.0378   (lower is better)
    Brier Score    : 0.0098   (lower is better)
    AUC-PR         : 0.9916   (higher is better)
    Top-1 Hit Rate : 1.000   (target > 0.35 pre / > 0.45 post)


### Feature Importance

In [16]:
def print_feature_importance(model, feature_cols, label, top_n=15):
    importance = model.feature_importances_
    pairs = sorted(
        zip(feature_cols, importance),
        key=lambda x: x[1], reverse=True
    )[:top_n]
    print(f"\n  {label}:")
    for feat, score in pairs:
        print(f"    {feat:<45} {score:.4f}")
 
print_feature_importance(pre_model,  pre_feats,  "Pre-Qualifying Model")
print_feature_importance(post_model, post_feats, "Post-Qualifying Model")


  Pre-Qualifying Model:
    driver_avg_quali_position_last_5              0.3144
    driver_avg_quali_position_last_3              0.0510
    team_avg_finish_last_5                        0.0382
    team_points_gap_to_leader                     0.0348
    driver_avg_finish_last_5                      0.0303
    races_remaining_this_season                   0.0249
    team_points_last_3                            0.0232
    driver_points_gap_to_leader                   0.0227
    driver_avg_quali_position_this_season         0.0224
    avg_finish_gap_to_teammate_this_season        0.0213
    team_dnf_rate_at_circuit                      0.0208
    driver_championship_position                  0.0203
    team_avg_finish_at_circuit_last5              0.0202
    driver_avg_quali_position_at_circuit_last5    0.0195
    team_avg_finish_last_3                        0.0171

  Post-Qualifying Model:
    driver_avg_quali_position_last_5              0.2249
    grid_position                    

### Save Models and Metadata

In [17]:
with open("saved_models/pre_qual_model.pkl",       "wb") as f: pickle.dump(pre_model,   f)

with open("saved_models/post_qual_model.pkl",      "wb") as f: pickle.dump(post_model,  f)

with open("saved_models/pre_qual_calibrated.pkl",  "wb") as f: pickle.dump(pre_cal,     f)

with open("saved_models/post_qual_calibrated.pkl", "wb") as f: pickle.dump(post_cal,    f)

In [18]:
# Save train medians for NaN fill at inference time in Flask API
# API must fill NaN using TRAINING medians, not new data
with open("saved_models/pre_qual_medians.pkl",  "wb") as f: pickle.dump(pre_medians,  f)

with open("saved_models/post_qual_medians.pkl", "wb") as f: pickle.dump(post_medians, f)

In [ ]:
# Save all metadata — Flask API loads this at startup
# LEAKAGE FIX: Now includes explicit train/calibration/test season split
feature_meta = {
    "pre_qual_features":        pre_feats,
    "post_qual_features":       post_feats,
    "identity_cols":            identity_cols,
    "target":                   target,
    "scale_pos_weight":         scale_pos_weight,
    "train_seasons":            pre_train_seasons,
    "calibration_season":       pre_cal_season,
    "test_season":              pre_test_season,
    "all_seasons":              seasons,
    "pre_qual_cv_metrics":      pre_cv,
    "post_qual_cv_metrics":     post_cv,
    "pre_qual_final_metrics":   pre_final_metrics,
    "post_qual_final_metrics":  post_final_metrics,
}
 
with open("saved_models/feature_columns.json", "w") as f:
    json.dump(feature_meta, f, indent=2)


### SAMPLE PREDICTION ON LAST RACE IN DATASET

In [ ]:
# Pick last race in test season
# LEAKAGE FIX: Sample from test set, which is held-out
last_round   = post_df[post_df["season"] == post_test_season]["round"].max()
last_circuit = post_df[
    (post_df["season"] == post_test_season) &
    (post_df["round"]  == last_round)
]["circuit_id"].iloc[0]
 
sample = post_df[
    (post_df["season"] == post_test_season) &
    (post_df["round"]  == last_round)
].copy()
 
if not sample.empty:
    # Pre-qual prediction
    X_pre_sample = sample[[c for c in pre_feats if c in sample.columns]].copy()
    X_pre_sample = X_pre_sample.fillna(pre_medians)
    pre_raw      = pre_cal.predict_proba(X_pre_sample)[:, 1]
    pre_norm     = pre_raw / pre_raw.sum()
 
    # Post-qual prediction
    X_post_sample = sample[[c for c in post_feats if c in sample.columns]].copy()
    X_post_sample = X_post_sample.fillna(post_medians)
    post_raw      = post_cal.predict_proba(X_post_sample)[:, 1]
    post_norm     = post_raw / post_raw.sum()
 
    sample["pre_qual_prob"]  = pre_norm
    sample["post_qual_prob"] = post_norm
 
    print(f"\n  Race: {last_circuit} {post_test_season} (Round {last_round})")
    print(f"  [Sample from HELD-OUT test set]")
    print(f"\n  {'Driver':<8} {'Team':<22} {'Pre-Qual':>10} {'Post-Qual':>10} {'Grid':>6} {'Result':>8}")
    print(f"  {'-' * 70}")
 
    display = sample[["driver", "team", "pre_qual_prob", "post_qual_prob",
                       "grid_position", target]].copy()
    display  = display.sort_values("post_qual_prob", ascending=False)
 
    for _, r in display.iterrows():
        result = "✓ WIN" if r[target] == 1 else ""
        grid   = int(r["grid_position"]) if pd.notna(r.get("grid_position")) else "-"
        print(
            f"  {str(r['driver']):<8} {str(r['team']):<22} "
            f"{r['pre_qual_prob']:>9.3f}  {r['post_qual_prob']:>9.3f}  "
            f"{str(grid):>5}  {result:>7}"
        )
 
    # Verify probabilities sum to 1
    print(f"\n  Pre-qual probs sum  : {pre_norm.sum():.6f}  (should be 1.0)")
    print(f"  Post-qual probs sum : {post_norm.sum():.6f}  (should be 1.0)")
 



  Race: Yas Island 2025 (Round 24)

  Driver   Team                     Pre-Qual  Post-Qual   Grid   Result
  ----------------------------------------------------------------------
  VER      Red Bull Racing            0.420      0.865      1    ✓ WIN
  NOR      McLaren                    0.119      0.049      2         
  PIA      McLaren                    0.251      0.049      3         
  ANT      Mercedes                   0.027      0.013     14         
  HAM      Ferrari                    0.000      0.013     16         
  LEC      Ferrari                    0.057      0.013      5         
  ALB      Williams                   0.000      0.000     17         
  ALO      Aston Martin               0.000      0.000      6         
  BEA      Haas F1 Team               0.000      0.000     11         
  HAD      Racing Bulls               0.000      0.000      9         
  BOR      Kick Sauber                0.000      0.000      7         
  COL      Alpine                    